In [16]:
import pandas as pd

In [17]:



path_book_data = "../dataset/readyToUse/book_data_processed_and_cleaned.xlsx"

book_data = pd.read_excel(path_book_data)

print(f"len(book_data) = {len(book_data)} ")





len(book_data) = 212403 


In [ ]:
path_book_reviews = "../dataset/book_reviews_sampled.xlsx"
book_reviews = pd.read_excel(path_book_reviews)
print(f"len(book_reviews) = {len(book_reviews)} ")

## Estrazione casuale per dataframe ridotti

In [5]:
# Estrai il 10% casuale
book_data_sample = book_data.sample(frac=0.1, random_state=42)
print(f"len(book_data_sample) = {len(book_data_sample)} ")

book_data_sample.to_excel("../dataset/book_data_processed_and_cleaned_small.xlsx", index=False)


len(book_data_sample) = 21240 


## Prove per spezzettamento dataframe

In [ ]:


def get_df_size_mb_pickle(df):
    """Calcola dimensione reale usando pickle"""
    return len(pickle.dumps(df)) / (1024 * 1024)

In [25]:
print(f"DataFrame size (pickle): {get_df_size_mb_pickle(book_data):.2f} MB")

get_df_size_mb_pickle(book_data)//50 + 1

DataFrame size (pickle): 165.67 MB


4.0

In [22]:

def extractChunksAndInsertIntoWeaviateProgressBar( df):
    num_partitions = int((get_df_size_mb_pickle(df) // 100) + 1)
    total_length = len(df)
    chunk_size = total_length // num_partitions

    print(f"Inizio inserimento in {num_partitions} chunk, {total_length} righe totali.")

    for i in range(num_partitions):
        start = i * chunk_size
        if i == num_partitions - 1:
            end = total_length
        else:
            end = (i + 1) * chunk_size

        df_chunked = df.iloc[start:end]

        print(f"Inserimento di {df_chunked.shape[0]} righe (righe {start} a {end})...")
        print("Inserito!")

        progress = (i + 1) / num_partitions * 100
        print(f"Chunk {i + 1}/{num_partitions} inserito ({progress:.1f}%).")

In [23]:
extractChunksAndInsertIntoWeaviateProgressBar(book_data)

Inizio inserimento in 2 chunk, 212403 righe totali.
Inserimento di 106201 righe (righe 0 a 106201)...
Inserito!
Chunk 1/2 inserito (50.0%).
Inserimento di 106202 righe (righe 106201 a 212403)...
Inserito!
Chunk 2/2 inserito (100.0%).


## Pulizia book_data.csv 
### gia fatta

In [19]:

##Gestione Colonna titolo:
# Elimina tutte le righe dove la colonna 'Title' è vuota o "N/A"
print(f"Righe prima della pulizia: {len(book_data)}")

# Rimuovi righe con Title vuoto, NaN o "N/A"
book_data = book_data[
    (book_data['Title'].notna()) & 
    (book_data['Title'] != 'N/A') & 
    (book_data['Title'] != '') &
    (book_data['Title'].str.strip() != '')
].copy()

print(f"Righe dopo la pulizia: {len(book_data)}")
print(f"Righe eliminate: {len(book_data) - len(book_data)}")

# Verifica che non ci siano più titoli vuoti
print(f"Titoli vuoti rimasti: {book_data['Title'].isna().sum()}")





Righe prima della pulizia: 212403
Righe dopo la pulizia: 212403
Righe eliminate: 0
Titoli vuoti rimasti: 0


In [20]:

##Gestione Valori Nulli

# Per ogni colonna di tipo 'object' (cioè stringhe/testo), sostituisci NaN con "N/A"
for col in book_data.select_dtypes(include="object").columns:
    book_data[col] = book_data[col].fillna("N/A")

#sostituisci NaN con 0 per numero recensioni
book_data["ratingsCount"] = book_data["ratingsCount"].fillna(0).astype(int)



            

In [21]:


# Gestione colonna authors:

import ast

# Prima controlliamo i formati attuali della colonna Authors
print("=== ANALISI COLONNA AUTHORS ===")
print(f"Tipo di dati: {book_data['authors'].dtype}")
print(f"Valori unici (primi 10): {book_data['authors'].unique()[:10]}")
print(f"Valori nulli: {book_data['authors'].isna().sum()}")

# Funzione per verificare se un valore è nel formato corretto per array
def check_author_format(author_value):
    if pd.isna(author_value) or author_value == "N/A":
        return "null_or_na"
    
    # Controlla se è già una lista
    if isinstance(author_value, list):
        return f"already_list_{len(author_value)}_elements"
    
    # Controlla se è una stringa che rappresenta una lista
    if isinstance(author_value, str):
        author_value = author_value.strip()
        
        if author_value.startswith('[') and author_value.endswith(']'):
            try:
                parsed = ast.literal_eval(author_value)
                if isinstance(parsed, list):
                    return f"valid_list_string_{len(parsed)}_elements"
            except:
                return "invalid_list_format"
        else:
            return "not_list_format"
    
    return "unknown_format"

# Applica il controllo
book_data['author_format_check'] = book_data['authors'].apply(check_author_format)

# Mostra i risultati del controllo
print("\n=== RISULTATI CONTROLLO FORMATO ===")
format_counts = book_data['author_format_check'].value_counts()
print(format_counts)

# Mostra alcuni esempi per ogni categoria
print("\n=== ESEMPI PER CATEGORIA ===")
for format_type in format_counts.index:
    print(f"\n{format_type.upper()}:")
    examples = book_data[book_data['author_format_check'] == format_type]['authors'].head(2)
    for idx, example in examples.items():
        print(f"  - {example}")

# Funzione per pulire e standardizzare gli autori (supporta N elementi)
def clean_authors(author_value):
    if pd.isna(author_value) or author_value == "N/A":
        return ['']
    
    if isinstance(author_value, list):
        # Se è già una lista, pulisci gli elementi (supporta qualsiasi numero)
        cleaned = [str(author).strip() for author in author_value if str(author).strip() and str(author).strip() != 'N/A']
        return cleaned if cleaned else ['']
    
    if isinstance(author_value, str):
        author_value = author_value.strip()
        
        # Se è nel formato stringa di lista
        if author_value.startswith('[') and author_value.endswith(']'):
            try:
                parsed = ast.literal_eval(author_value)
                if isinstance(parsed, list):
                    cleaned = [str(author).strip() for author in parsed if str(author).strip() and str(author).strip() != 'N/A']
                    return cleaned if cleaned else ['']
            except:
                pass
        
        # Se è una singola stringa, mettila in una lista
        if author_value and author_value != "N/A":
            return [author_value]
    
    return ['']

# Applica la pulizia
book_data['authors_cleaned'] = book_data['authors'].apply(clean_authors)

# Verifica finale - mostra esempi con diversi numeri di autori
print("\n=== VERIFICA FINALE ===")
print("Esempi di autori puliti (tutti i formati):")

# Raggruppa per numero di autori
author_lengths = book_data['authors_cleaned'].apply(len)
for length in sorted(author_lengths.unique())[:5]:  # Mostra primi 5 tipi diversi
    examples = book_data[author_lengths == length]['authors_cleaned'].head(2)
    print(f"\nLibri con {length} autor{'e' if length == 1 else 'i'}:")
    for idx, authors in examples.items():
        print(f"  - {authors}")

# Sostituisci la colonna originale
book_data['authors'] = book_data['authors_cleaned']
book_data.drop(['author_format_check', 'authors_cleaned'], axis=1, inplace=True)

print(f"\nPulizia completata! Tutti gli autori sono ora nel formato lista con N elementi.")
print(f"Distribuzione numero autori per libro:")
author_counts = book_data['authors'].apply(len).value_counts().sort_index()
print(author_counts.head(10))


=== ANALISI COLONNA AUTHORS ===
Tipo di dati: object
Valori unici (primi 10): ["['Julie Strain']" "['Philip Nel']" "['David R. Ray']"
 "['Veronica Haddon']" "['Edward Long']" "['Everett Ferguson']"
 "['Miriam Allen De Ford']" "['Lee Blessing']" "['Mary Fabyan Windeatt']"
 "['Steven Wardell']"]
Valori nulli: 0

=== RISULTATI CONTROLLO FORMATO ===
author_format_check
valid_list_string_1_elements      145610
null_or_na                         31413
valid_list_string_2_elements       26810
valid_list_string_3_elements        6180
valid_list_string_4_elements        1511
valid_list_string_5_elements         464
valid_list_string_6_elements         174
valid_list_string_7_elements          82
valid_list_string_8_elements          42
valid_list_string_9_elements          29
valid_list_string_10_elements         18
valid_list_string_11_elements         12
valid_list_string_12_elements          9
valid_list_string_13_elements          7
valid_list_string_17_elements          4
valid_list_string

### gestione colonna publishedDate:

In [22]:


from datetime import datetime
import re

# Prima controlliamo i formati attuali della colonna publishedDate
print("=== ANALISI COLONNA PUBLISHEDDATE ===")
print(f"Tipo di dati: {book_data['publishedDate'].dtype}")
print(f"Valori unici (primi 15): {book_data['publishedDate'].unique()[:15]}")
print(f"Valori nulli: {book_data['publishedDate'].isna().sum()}")

# Funzione per verificare e categorizzare i formati di data
def check_date_format(date_value):
    if pd.isna(date_value) or date_value == "N/A" or date_value == "":
        return "null_or_na"
    
    date_str = str(date_value).strip()
    
    # Formato Weaviate (ISO 8601 con timezone Z)
    if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$', date_str):
        return "weaviate_format_correct"
    
    # Formato ISO senza timezone
    if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$', date_str):
        return "iso_without_timezone"
    
    # Anno soltanto (es: 2020, 1995)
    if re.match(r'^\d{4}$', date_str):
        return "year_only"
    
    # Formato completo YYYY-MM-DD (es: 2022-02-23)
    if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
        return "full_date_iso"
    
    # Formato YYYY-MM (es: 2022-02)
    if re.match(r'^\d{4}-\d{2}$', date_str):
        return "year_month"
    
    # Altri formati possibili
    if re.match(r'^\d{1,2}/\d{1,2}/\d{4}$', date_str):
        return "mm_dd_yyyy"
    
    if re.match(r'^\d{4}/\d{1,2}/\d{1,2}$', date_str):
        return "yyyy_mm_dd_slash"
    
    return "unknown_format"


# Funzione per standardizzare le date nel formato ISO 8601 per Weaviate
def clean_published_date(date_value):
    if pd.isna(date_value) or date_value == "N/A" or date_value == "":
        return "1900-01-01T00:00:00Z"  # Data standard per valori nulli
    
    date_str = str(date_value).strip()
    
    try:
        # Anno soltanto (es: 2020) -> 2020-01-01T00:00:00Z
        if re.match(r'^\d{4}$', date_str):
            return f"{date_str}-01-01T00:00:00Z"
        
        # Formato completo YYYY-MM-DD -> YYYY-MM-DDTHH:MM:SSZ
        if re.match(r'^\d{4}-\d{2}-\d{2}$', date_str):
            return f"{date_str}T00:00:00Z"
        
        # Formato YYYY-MM -> YYYY-MM-01T00:00:00Z
        if re.match(r'^\d{4}-\d{2}$', date_str):
            return f"{date_str}-01T00:00:00Z"
        
        # Formato MM/DD/YYYY
        if re.match(r'^\d{1,2}/\d{1,2}/\d{4}$', date_str):
            parts = date_str.split('/')
            month = parts[0].zfill(2)
            day = parts[1].zfill(2)
            year = parts[2]
            return f"{year}-{month}-{day}T00:00:00Z"
        
        # Formato YYYY/MM/DD
        if re.match(r'^\d{4}/\d{1,2}/\d{1,2}$', date_str):
            parts = date_str.split('/')
            year = parts[0]
            month = parts[1].zfill(2)
            day = parts[2].zfill(2)
            return f"{year}-{month}-{day}T00:00:00Z"
        
        # Formato già corretto per Weaviate (ISO 8601 con timezone)
        if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z$', date_str):
            return date_str  # Già nel formato corretto
        
        # Formato ISO senza timezone -> aggiungi Z
        if re.match(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}$', date_str):
            return f"{date_str}Z"
        
        # Prova a parsare con pandas
        try:
            parsed_date = pd.to_datetime(date_str)
            return parsed_date.strftime("%Y-%m-%dT%H:%M:%SZ")
        except:
            pass
        
    except Exception as e:
        print(f"Errore nel parsing della data: {date_str} - {e}")
    
    # Per formati non riconosciuti, usa una data standard irrealistica
    print(f"Formato non riconosciuto: {date_str} -> usando data standard")
    return "1900-01-01T00:00:00Z"


# Applica il controllo
book_data['date_format_check'] = book_data['publishedDate'].apply(check_date_format)

# Mostra i risultati del controllo
print("\n=== RISULTATI CONTROLLO FORMATO DATE ===")
format_counts = book_data['date_format_check'].value_counts()
print(format_counts)

# Mostra alcuni esempi per ogni categoria
print("\n=== ESEMPI PER CATEGORIA ===")
for format_type in format_counts.index:
    print(f"\n{format_type.upper()}:")
    examples = book_data[book_data['date_format_check'] == format_type]['publishedDate'].head(3)
    for idx, example in examples.items():
        print(f"  - {example}")



# Applica la pulizia
book_data['publishedDate_cleaned'] = book_data['publishedDate'].apply(clean_published_date)

# Verifica finale
print("\n=== VERIFICA FINALE ===")
print("Esempi di date standardizzate:")
for i in range(10):
    if i < len(book_data):
        original = book_data.iloc[i]['publishedDate']
        cleaned = book_data.iloc[i]['publishedDate_cleaned']
        format_type = book_data.iloc[i]['date_format_check']
        print(f"Originale: {original} ({format_type}) -> Pulito: {cleaned}")

# Conta quante date sono state convertite con successo
successful_conversions = book_data['publishedDate_cleaned'].notna().sum()
total_dates = len(book_data)
print(f"\nConversioni riuscite: {successful_conversions}/{total_dates} ({successful_conversions/total_dates*100:.1f}%)")

# Sostituisci la colonna originale
book_data['publishedDate'] = book_data['publishedDate_cleaned']
book_data.drop(['date_format_check', 'publishedDate_cleaned'], axis=1, inplace=True)

print(f"\nPulizia completata! Tutte le date sono ora nel formato ISO 8601 per Weaviate.")

=== ANALISI COLONNA PUBLISHEDDATE ===
Tipo di dati: object
Valori unici (primi 15): ['1996' '2005-01-01' '2000' '2005-02' '2003-03-01' '1960' '1988'
 '2009-01-01' '1995' '1994-02-17' '2005-07' '2018-11-06' '2012-12-06'
 '2018-02-27' '2009']
Valori nulli: 0

=== RISULTATI CONTROLLO FORMATO DATE ===
date_format_check
year_only                  90987
full_date_iso              84804
null_or_na                 25305
year_month                 10973
unknown_format               291
weaviate_format_correct       43
Name: count, dtype: int64

=== ESEMPI PER CATEGORIA ===

YEAR_ONLY:
  - 1996
  - 2000
  - 1996

FULL_DATE_ISO:
  - 2005-01-01
  - 2003-03-01
  - 2009-01-01

NULL_OR_NA:
  - N/A
  - N/A
  - N/A

YEAR_MONTH:
  - 2005-02
  - 2005-07
  - 2000-01

UNKNOWN_FORMAT:
  - 1963*
  - 19??
  - 1973*

WEAVIATE_FORMAT_CORRECT:
  - 2022-06-22T22:59:00Z
  - 2022-05-09T22:59:00Z
  - 2020-08-17T23:42:12Z
Formato non riconosciuto: 1963* -> usando data standard
Formato non riconosciuto: 19?? -> usando

In [15]:
# Sostituzione rapida di valori null con data standard Weaviate
book_data['publishedDate'] = book_data['publishedDate'].fillna("1900-01-01T00:00:00Z")
book_data['publishedDate'] = book_data['publishedDate'].replace("N/A", "1900-01-01T00:00:00Z")
book_data['publishedDate'] = book_data['publishedDate'].replace("", "1900-01-01T00:00:00Z")

print(f"Date standardizzate: {(book_data['publishedDate'] == '1900-01-01T00:00:00Z').sum()}")

Date standardizzate: 25718


# ----------------------------------------

In [23]:
book_data.to_excel("book_data_processed_and_cleaned.xlsx", index=False)


# --------------------------------------------------

### Estrazione dati da book_ratings_full.csv

#### gia fatta, ma riutilizzabile per estrarre diverse taglie!

In [ ]:
# Estrai 1 milione di righe casuali (senza ripetizione)
book_reviews_samples = book_reviews.sample(n=1_000_000, random_state=42)

# Se vuoi salvarlo in un nuovo file
book_reviews_samples.to_csv("book_reviews_sampled.csv")


## Pulizia book_ratings